# D-FINE M training on 1080x1080 sliced dataset

This notebook clones the workspace, mounts Google Drive, extracts a YOLO-format dataset zip, resizes source frames to 1920x1080, slices them into 1080x1080 crops, converts the sliced YOLO dataset to COCO, downloads D-FINE M pretrained weights, and starts training.

Expected dataset shape inside the zip:

```text
Dataset/
  Train/
    images/
    labels/
  Validation/
    images/
    labels/
```


In [ ]:
from pathlib import Path

# Required: set this to your workspace repo URL before running all cells.
REPO_URL = 'https://github.com/YOUR_USERNAME/obj-det-ws.git'
REPO_BRANCH = ''  # Optional. Leave empty for the default branch.

# Required: Google Drive zip path after drive.mount('/content/drive').
DATASET_ZIP_DRIVE_PATH = '/content/drive/MyDrive/Dataset.zip'

# Training outputs are written to Drive. Prepared sliced dataset stays local.
OUTPUT_DRIVE_DIR = '/content/drive/MyDrive/dfine_runs/dfine_hgnetv2_m_datasetv1_sliced_1080'

# Weights & Biases. Leave WANDB_API_KEY empty to use Colab Secret named WANDB_API_KEY or interactive login.
USE_WANDB = True
WANDB_PROJECT = 'obj-det-ws-dfine'
WANDB_RUN_NAME = 'dfine_hgnetv2_m_datasetv1_sliced_1080_a100'
WANDB_ENTITY = ''
WANDB_API_KEY = ''

WORKSPACE_DIR = Path('/content/obj-det-ws')
LOCAL_DATASET_EXTRACT_DIR = Path('/content/dataset_raw')

CLASSES = ['arac', 'insan', 'uap', 'uai']
RESIZE_TO = '1920x1080'
CROP_SIZE = '1080x1080'
OVERLAP = 0.0

# A100 40GB defaults. D-FINE still resizes training inputs to 640x640 via the repo config.
EPOCHS = 132
TOTAL_BATCH_SIZE = 32
VAL_BATCH_SIZE = 64
NUM_WORKERS = 8
SEED = 0

PRETRAINED_URL = 'https://github.com/Peterande/storage/releases/download/dfinev1.0/dfine_m_obj365.pth'


In [ ]:
import os
import shutil
import subprocess
import sys

from google.colab import drive, userdata


def run(command, cwd=None):
    printable = ' '.join(str(part) for part in command)
    print(f'$ {printable}')
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)


if 'YOUR_USERNAME' in REPO_URL:
    raise ValueError('Set REPO_URL in the parameter cell before running the notebook.')

drive.mount('/content/drive')

if WORKSPACE_DIR.exists():
    shutil.rmtree(WORKSPACE_DIR)

clone_command = ['git', 'clone', '--depth', '1', '--recurse-submodules', '--shallow-submodules']
if REPO_BRANCH:
    clone_command.extend(['--branch', REPO_BRANCH])
clone_command.extend([REPO_URL, str(WORKSPACE_DIR)])
run(clone_command)

run([sys.executable, '-m', 'pip', 'install', '-q', '-r', WORKSPACE_DIR / 'models/D-FINE/requirements.txt'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'])
run([sys.executable, '-m', 'py_compile', WORKSPACE_DIR / 'tools/slice_dataset.py', WORKSPACE_DIR / 'tools/yolo_to_coco.py'])


In [ ]:
def get_colab_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


if USE_WANDB:
    import wandb

    api_key = WANDB_API_KEY or get_colab_secret('WANDB_API_KEY')
    if api_key:
        wandb.login(key=api_key)
    else:
        wandb.login()

    os.environ['WANDB_PROJECT'] = WANDB_PROJECT
    os.environ['WANDB_NAME'] = WANDB_RUN_NAME
    if WANDB_ENTITY:
        os.environ['WANDB_ENTITY'] = WANDB_ENTITY

    print(f'W&B enabled: project={WANDB_PROJECT}, run={WANDB_RUN_NAME}')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('W&B disabled')


In [ ]:
import zipfile


def find_split(dataset_root, names):
    for name in names:
        candidate = dataset_root / name
        if (candidate / 'images').is_dir() and (candidate / 'labels').is_dir():
            return candidate
    return None


def find_yolo_dataset_root(search_root):
    candidates = [search_root]
    candidates.extend(path for path in search_root.rglob('*') if path.is_dir())
    for candidate in candidates:
        train_split = find_split(candidate, ['Train', 'train'])
        val_split = find_split(candidate, ['Validation', 'validation', 'Val', 'val', 'Valid', 'valid'])
        if train_split is not None and val_split is not None:
            return candidate, train_split, val_split
    raise FileNotFoundError('Could not find Dataset/Train/{images,labels} and Dataset/Validation/{images,labels}.')


dataset_source = Path(DATASET_ZIP_DRIVE_PATH)
if not dataset_source.exists():
    raise FileNotFoundError(f'Dataset path not found: {dataset_source}')

if LOCAL_DATASET_EXTRACT_DIR.exists():
    shutil.rmtree(LOCAL_DATASET_EXTRACT_DIR)
LOCAL_DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if dataset_source.is_dir():
    dataset_search_root = dataset_source
else:
    if dataset_source.suffix.lower() != '.zip':
        raise ValueError('DATASET_ZIP_DRIVE_PATH must be a .zip file or an extracted dataset directory.')
    with zipfile.ZipFile(dataset_source) as archive:
        archive.extractall(LOCAL_DATASET_EXTRACT_DIR)
    dataset_search_root = LOCAL_DATASET_EXTRACT_DIR

dataset_root, train_source, val_source = find_yolo_dataset_root(dataset_search_root)
print(f'dataset_root: {dataset_root}')
print(f'train_source: {train_source}')
print(f'val_source: {val_source}')


In [ ]:
import json


SLICED_YOLO_ROOT = WORKSPACE_DIR / 'datasets/_sliced_yolo_datasetv1_1080'
FINAL_COCO_ROOT = WORKSPACE_DIR / 'datasets/datasetv1_sliced_1080'

for path in [SLICED_YOLO_ROOT, FINAL_COCO_ROOT]:
    if path.exists():
        shutil.rmtree(path)


def copy_images_to_coco_root(sliced_split_root, final_split_root):
    final_split_root.mkdir(parents=True, exist_ok=True)
    shutil.copytree(sliced_split_root / 'images', final_split_root, dirs_exist_ok=True)


def convert_split(split_name, source_split, annotation_name):
    sliced_split_root = SLICED_YOLO_ROOT / split_name
    final_split_root = FINAL_COCO_ROOT / split_name
    final_annotation_path = final_split_root / annotation_name

    run([
        sys.executable,
        WORKSPACE_DIR / 'tools/slice_dataset.py',
        source_split,
        '-o', sliced_split_root,
        '--format', 'yolo',
        '--resize-to', RESIZE_TO,
        '--crop-size', CROP_SIZE,
        '--overlap', str(OVERLAP),
    ])

    final_split_root.mkdir(parents=True, exist_ok=True)
    run([
        sys.executable,
        WORKSPACE_DIR / 'tools/yolo_to_coco.py',
        sliced_split_root,
        '-o', final_annotation_path,
        '--classes', *CLASSES,
    ])
    copy_images_to_coco_root(sliced_split_root, final_split_root)
    return final_annotation_path, final_split_root


train_json, train_image_root = convert_split('train', train_source, 'train.json')
val_json, val_image_root = convert_split('val', val_source, 'val.json')


def validate_coco(annotation_path, image_root):
    coco = json.loads(annotation_path.read_text())
    category_ids = [category['id'] for category in coco['categories']]
    category_names = [category['name'] for category in coco['categories']]
    if category_ids != list(range(len(CLASSES))) or category_names != CLASSES:
        raise ValueError(f'Unexpected categories in {annotation_path}: {coco["categories"]}')
    if not coco['images']:
        raise ValueError(f'No images in {annotation_path}')
    images_by_id = {image['id']: image for image in coco['images']}
    for image in coco['images'][:20]:
        if not (image_root / image['file_name']).is_file():
            raise FileNotFoundError(image_root / image['file_name'])
    for annotation in coco['annotations'][:1000]:
        image = images_by_id[annotation['image_id']]
        x, y, width, height = annotation['bbox']
        if x < 0 or y < 0 or width <= 0 or height <= 0:
            raise ValueError(f'Invalid bbox in {annotation_path}: {annotation}')
        if x + width > image['width'] + 1e-6 or y + height > image['height'] + 1e-6:
            raise ValueError(f'Bbox outside image in {annotation_path}: {annotation}')
    print(f'{annotation_path}: {len(coco["images"])} images, {len(coco["annotations"])} annotations')


validate_coco(train_json, train_image_root)
validate_coco(val_json, val_image_root)


In [ ]:
WEIGHTS_DIR = WORKSPACE_DIR / 'weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PRETRAINED_PATH = WEIGHTS_DIR / 'dfine_m_obj365.pth'

if not PRETRAINED_PATH.exists():
    run(['wget', '-q', '--show-progress', '-O', PRETRAINED_PATH, PRETRAINED_URL])
else:
    print(f'Using existing checkpoint: {PRETRAINED_PATH}')

print(f'pretrained: {PRETRAINED_PATH}')


In [ ]:
CONFIG_PATH = 'configs/d-fine/dfine_hgnetv2_m_datasetv1_sliced_1080.yml'
OUTPUT_DIR = Path(OUTPUT_DRIVE_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_COMMAND = [
    sys.executable,
    'scripts/d-fine/train.py',
    '--devices', '0',
    '--config', CONFIG_PATH,
    '--tuning', str(PRETRAINED_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--seed', str(SEED),
    '--update',
    f'epochs={EPOCHS}',
    f'train_dataloader.total_batch_size={TOTAL_BATCH_SIZE}',
    f'val_dataloader.total_batch_size={VAL_BATCH_SIZE}',
    f'train_dataloader.num_workers={NUM_WORKERS}',
    f'val_dataloader.num_workers={NUM_WORKERS}',
    f'use_wandb={str(USE_WANDB)}',
    f'project_name={WANDB_PROJECT}',
    f'exp_name={WANDB_RUN_NAME}',
]

run([*TRAIN_COMMAND, '--dry-run'], cwd=WORKSPACE_DIR)


In [ ]:
run(TRAIN_COMMAND, cwd=WORKSPACE_DIR)
